In [1]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

namespace = ("users",)
key = "user_101"

store.put(
    ("users",),
    "user_101",
    {
        "name": "Asadullah",
        "language": "Python",
        "preference": "Beginner-friendly explanations"
    }
)

memory = store.get(
    ("users",),
    "user_101"
)

print(memory.value)

store.put(
    ("users",),
    "user_101",
    {
        "name": "Asadullah",
        "language": "Python",
        "preference": "Advanced explanations"
    }
)


{'name': 'Asadullah', 'language': 'Python', 'preference': 'Beginner-friendly explanations'}


In [7]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

@dataclass
class UserContext:
    user_id: str

In [8]:
@tool
def save_user_memory(
    name: str,
    language: str,
    runtime: ToolRuntime[UserContext]
) -> str:
    """Save the user's name and preferred programming language."""

    runtime.store.put(
        ("users",),
        runtime.context.user_id,
        {
            "name": name,
            "language": language
        }
    )

    return "User information saved successfully."

@tool
def get_user_memory(
    runtime: ToolRuntime[UserContext]
) -> str:
    """Retrieve the current user's saved information."""

    memory = runtime.store.get(
        ("users",),
        runtime.context.user_id
    )

    if memory:
        return str(memory.value)

    return "No saved information found."

In [9]:
model = ChatGroq(model="openai/gpt-oss-120b")
agent = create_agent(
    model=model,
    tools=[
        save_user_memory,
        get_user_memory
    ],
    store=store,
    context_schema=UserContext,
    system_prompt="""
    You are a helpful personal assistant.

    When the user shares their name or programming preferences,
    save that information using the save_user_memory tool.

    When the user asks about previously saved information,
    retrieve it using the get_user_memory tool.

    Never invent saved user information.
    """
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Asadullah and I prefer Python."
            }
        ]
    },
    context=UserContext(
        user_id="user_101"
    )
)

print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What programming language do I prefer?"
            }
        ]
    },
    {
        "configurable": {
            "thread_id": "conversation_2"
        }
    },
    context=UserContext(
        user_id="user_101"
    )
)

print(response["messages"][-1].content)

In [11]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[
        save_user_memory,
        get_user_memory
    ],
    checkpointer=checkpointer,
    store=store,
    context_schema=UserContext
)

In [13]:
# !pip install -U langgraph-checkpoint-postgres "psycopg[binary]"

In [ ]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langgraph.checkpoint.postgres import PostgresSaver
from dotenv import load_dotenv
import os 
load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b"
)

DB_URI = os.getenv("DATABASE_URL")
print(DB_URI)

In [23]:

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:

    checkpointer.setup()

    agent = create_agent(
        model=model,
        tools=[],
        checkpointer=checkpointer
    )

    config = {
        "configurable": {
            "thread_id": "user_101"
        }
    }

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "My name is Asadullah."
                }
            ]
        },
        config=config
    )

    print(response["messages"][-1].content)

Nice to meet you, Asadullah! How can I assist you today?


In [24]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

from langgraph.checkpoint.postgres import PostgresSaver

model = ChatGroq(
    model="openai/gpt-oss-120b"
)

DB_URI = os.getenv("DATABASE_URL")

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:

    checkpointer.setup()

    agent = create_agent(
        model=model,
        tools=[],
        checkpointer=checkpointer
    )

    config = {
        "configurable": {
            "thread_id": "user_101"
        }
    }

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "My name is John."
                }
            ]
        },
        config=config
    )

    print(response["messages"][-1].content)

Got it—nice to meet you, John! How can I help you today?


In [27]:
from langgraph.store.postgres import PostgresStore
store = PostgresStore.from_conn_string(DB_URI)

with PostgresStore.from_conn_string(DB_URI) as store:

    store.setup()

    store.put(
        ("users",),
        "user_101",
        {
            "name": "Asadullah",
            "language": "Python",
            "preference": "Beginner-friendly explanations"
        }
    )

In [28]:
with PostgresStore.from_conn_string(DB_URI) as store:

    store.setup()

    memory = store.get(
        ("users",),
        "user_101"
    )

    if memory:
        print(memory.value)

{'name': 'Asadullah', 'language': 'Python', 'preference': 'Beginner-friendly explanations'}


In [29]:
with PostgresStore.from_conn_string(DB_URI) as store:

    store.setup()

    store.put(
        ("users",),
        "user_101",
        {
            "name": "Asadullah",
            "language": "Python",
            "preference": "Advanced explanations"
        }
    )

In [30]:
from dataclasses import dataclass

from langchain.tools import tool, ToolRuntime


@dataclass
class UserContext:
    user_id: str


@tool
def save_memory(
    name: str,
    programming_language: str,
    runtime: ToolRuntime[UserContext]
) -> str:
    """Save the user's name and preferred programming language."""

    runtime.store.put(
        ("users",),
        runtime.context.user_id,
        {
            "name": name,
            "programming_language": programming_language
        }
    )

    return "User information saved successfully."


@tool
def get_memory(
    runtime: ToolRuntime[UserContext]
) -> str:
    """Retrieve the current user's saved information."""

    memory = runtime.store.get(
        ("users",),
        runtime.context.user_id
    )

    if memory:
        return str(memory.value)

    return "No saved information found."

In [31]:
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

from langchain.agents import create_agent
from langchain_groq import ChatGroq

from dotenv import load_dotenv
import os

load_dotenv()

DB_URI = os.getenv("DATABASE_URL")

model = ChatGroq(
    model="openai/gpt-oss-120b"
)

In [32]:
with (
    PostgresSaver.from_conn_string(DB_URI) as checkpointer,
    PostgresStore.from_conn_string(DB_URI) as store
):

    # Initialize database schemas
    checkpointer.setup()
    store.setup()

    # Create agent
    agent = create_agent(
        model=model,
        tools=[save_memory, get_memory],
        checkpointer=checkpointer,
        store=store,
        context_schema=UserContext,
        system_prompt="""
        You are a helpful personal AI assistant.

        When users share their name and programming
        language preferences, save them using save_memory.

        When users ask about their saved information,
        retrieve it using get_memory.

        Never invent user information.
        """
    )

    # First conversation
    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "My name is Asadullah and I prefer Python."
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": "conversation_1"
            }
        },
        context=UserContext(user_id="user_101")
    )

    print(response["messages"][-1].content)

    # Second conversation
    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What programming language do I prefer?"
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": "conversation_2"
            }
        },
        context=UserContext(user_id="user_101")
    )

    print(response["messages"][-1].content)

Got it! I've saved your name as **Asadullah** and your preferred programming language as **Python**. If you need to see or update this information later, just let me know.
You mentioned that your preferred programming language is **Python**.
